In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_openai import  ChatOpenAI
from langchain_anthropic import ChatAnthropic

llm_client= ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.7,
)

In [ ]:
from langchain.chat_models import  init_chat_model
import os

llm_client= init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider=os.getenv("MODEL_PROVIDER"),
    temperature=0.7,
)

In [ ]:
response= llm_client.invoke("Tell me about npci in 2 bullet points")
response

In [ ]:
from IPython.display import display, Markdown

Markdown(response.content)

In [ ]:
from langchain.messages import  HumanMessage,SystemMessage,AIMessage

messages=[
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Tell me about npci in 2 bullet points")
]

llm_client.invoke(messages)

In [ ]:
messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell me about npci in 2 bullet points"}
]

llm_client.invoke(messages)

In [ ]:
llm_client.invoke("Tell me about india in 100 bullet points")

In [ ]:
for message in llm_client.stream("Tell me about india in 10 bullet points"):
    print(message.content , end="")

In [ ]:
messages =[
    "tell me about elon musk in 2 bullet points",
    "tell me about bill gates in 2 bullet points",
    "tell me about steve jobs in 2 bullet points"
]

llm_client.batch(messages)

In [ ]:
response = llm_client.invoke("Tell me about elon musk , his education, his achievements and his companies. Give me response in json format ")
response.content

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
parser.invoke(response)

In [ ]:
chain = llm_client | parser

chain.invoke("Tell me about elon musk , his education, his achievements and his companies. Give me response in json format ")

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    template="""
You are a helpful assistant who can transate text from {source_language} to {target_language}.
Translate {text}
"""
)
template

In [ ]:
template.invoke({
    "source_language": "English",
    "target_language": "French",
    "text": "Hello, how are you?"
})

In [ ]:
chain =  template | llm_client

chain.invoke({
    "source_language": "English",
    "target_language": "French",
    "text": "Hello, how are you?"
})

In [ ]:
template = PromptTemplate(
    template="""
Tell me about {person}, his education, his achievements and his companies. Give me response in json format

""")

chain = template | llm_client | parser

chain.invoke({
    "person": "Elon Musk"
})
 

In [ ]:
from langchain_core.prompts import  ChatPromptTemplate
from langchain_core.output_parsers import  StrOutputParser

template = ChatPromptTemplate(
    [
        ("system","you are an expert in {subject}"),
        ("user","Tell me about {topic} in 2 bullet points")
    ]
)

chain = template | llm_client | StrOutputParser()

chain.invoke({
    "subject": "technology",
    "topic": "blockchain"
})

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Person(BaseModel):
    name: str = Field(..., description="Name of the person")
    education: str = Field(..., description="Education of the person")
    achievements: str = Field(..., description="Achievements of the person")
    companies: str = Field(..., description="Companies founded by the person")
    age: int = Field(..., description="Age of the person in days")

parser = PydanticOutputParser(pydantic_object=Person)

parser.get_format_instructions()

In [38]:
from langchain_core.output_parsers import JsonOutputParser

chain = llm_client | parser

In [39]:
response= chain.invoke(
    f"""
    Tell me about elon musk , his education, his achievements and his companies.
    {parser.get_format_instructions()}
    

"""
)
response

Person(name='Elon Musk', education="Attended Queen's University in Canada for two years, then transferred to the University of Pennsylvania where he earned dual bachelor's degrees in Physics and Economics.", achievements='Co-founded Zip2, PayPal; founded SpaceX, Tesla, Neuralink, The Boring Company; developed reusable rockets; advanced electric vehicles; contributed to solar energy adoption; proposed Hyperloop concept.', companies='Zip2, PayPal, SpaceX, Tesla, Neuralink, The Boring Company, OpenAI (co-founder), SolarCity (merged with Tesla).', age=24223)